In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import classification_report, confusion_matrix

BASE_DIR = Path(r"C:\Magisterka")

CSV_PATH = BASE_DIR / "luad_patches_with_splits_all_classes.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device 

device(type='cpu')

In [2]:
df = pd.read_csv(CSV_PATH)
print("Columns in the dataset:", df.columns.tolist())

df["dominant_label_name"].value_counts()

Columns in the dataset: ['filename', 'image_path', 'mask_path', 'total_pixels', 'dominant_label_id', 'dominant_label_name', 'background_pix', 'cribriform_pix', 'micropapillary_pix', 'solid_pix', 'papillary_pix', 'acinar_pix', 'lepidic_pix', 'split']


dominant_label_name
background        483
papillary          85
solid              85
cribriform         35
lepidic            21
micropapillary     15
acinar              7
Name: count, dtype: int64

In [3]:
df_train = df[df["split"] == "train"].reset_index(drop=True)
df_val = df[df["split"] == "val"].reset_index(drop=True)
df_test = df[df["split"] == "test"].reset_index(drop=True)

len(df_train), len(df_val), len(df_test)

(511, 110, 110)

In [8]:
#Data Loaders and dataset 

IMG_SIZE = 224
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize( mean=[0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize( mean=[0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225]),
])


In [9]:
class PatchDataset(Dataset):
    def __init__(self, df_subset: pd.DataFrame, transform=None):
        self.df = df_subset.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["patch_path"]
        label_id = int(row["dominant_label_id"])
        
        img = Image.open(img_path).convert("RGB")
        
        if self.transform:
            img = self.transform(img)
        
        return img, label_id

In [10]:
BATCH_SIZE = 16
train_dataset = PatchDataset(df_train, transform=train_transform)
val_dataset = PatchDataset(df_val, transform=eval_transform)
test_dataset = PatchDataset(df_test, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

len(train_loader), len(val_loader), len(test_loader)

(32, 7, 7)

In [ ]:
NUM_CLASSES = 7 
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, NUM_CLASSES)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)


0.6%

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Mateusz/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%



In [14]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct_preds = 0
    total_samples = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct_preds += (preds == labels).sum().item()
        total_samples += labels.size(0)

    epoch_loss = running_loss / total_samples
    epoch_acc = correct_preds / total_samples

    return epoch_loss, epoch_acc

@torch.no_grad()
def evaluate_model(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_preds = 0
    total_samples = 0
    
    all_labels = []
    all_preds = []

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct_preds += (preds == labels).sum().item()
        total_samples += labels.size(0)
        
        all_labels.append(labels.cpu().numpy())
        all_preds.append(preds.cpu().numpy())

    epoch_loss = running_loss / total_samples
    epoch_acc = correct_preds / total_samples

    return epoch_loss, epoch_acc, all_labels, all_preds

In [ ]:
EPOCHS = 20
best_val_acc = 0.0
best_model_path = BASE_DIR / "rasnet18_baseline_best.pth"

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_labels, val_preds = evaluate_model(model, val_loader, criterion, device)

    print(f"Epoch {epoch}/{EPOCHS}:")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.3f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.3f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"Best model saved with Val Acc: {best_val_acc:.4f}")

print("Training complete. The best va;_acc", best_val_acc)

C:\Users\Mateusz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=device))

test_loss, test_acc, test_labels, test_preds = evaluate_model(model, test_loader, criterion, device)

print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.3f}")


In [ ]:
label_id_to_name = df.set_index("dominant_label_id")["dominant_label_name"].to_dict()
ordered_ids = sorted(label_id_to_name.keys())
target_names = [label_id_to_name[i] for i in ordered_ids]

print(classification_report(
    test_labels,
    test_preds,
    labels=ordered_ids,
    target_names=target_names,
    digits=3
))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(test_labels, test_preds, labels=ordered_ids)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=target_names, yticklabels=target_names,)

plt.xlabel('Predicted Label', fontsize=14)
plt.ylabel('True Label', fontsize=14)
plt.title('Confusion Matrix', fontsize=16)
plt.tight_layout()
plt.show()